In [1]:
from pathlib import Path
import pandas as pd

from phd_visualizations.regression import regression_plot
from phd_visualizations.calculations import SupportedInstruments 
from phd_visualizations import save_figure

import combined_cooler
import matlab

%reload_ext autoreload
%autoreload 2

data_path = Path("../assets")

df = pd.read_csv(data_path / "data.csv")
df


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,Rs,Qreleased,Qabsorbed,Qtransferred_option_1,Qtransferred_option_2,Qtransferred_option_3,Qtransferred_option_4,Qtransferred_option_5,Qtransferred_option_6,Qtransferred_option_7
0,02-Dec-2024,43.324459,20.923333,42.056077,17.998552,30.498067,1.947239,3.261430,42.871410,40.086938,...,0.000,202.144133,198.293522,166.373224,304.024595,178.053259,250.033088,187.296809,136.357530,119.725025
1,02-Dec-2024,42.869820,20.486687,52.055926,17.999154,29.898601,0.595016,2.015121,42.321919,39.623784,...,0.000,206.633807,200.773128,167.234472,303.887404,178.866884,252.533100,189.169538,138.698340,120.854515
2,19-Apr-2024,43.900000,16.440000,51.500000,24.000000,33.190000,0.320000,4.360000,43.260000,40.800000,...,0.954,221.957054,209.201968,172.342256,297.035023,183.690735,220.605447,165.252914,125.739895,119.462572
3,19-Apr-2024,43.640000,16.800000,50.740000,24.000000,33.160000,0.360000,4.760000,42.890000,40.620000,...,0.982,220.683015,205.023972,167.991007,292.655963,179.442931,215.555310,161.469916,122.918057,116.711260
4,10-Jan-2025,43.082367,20.522458,22.689454,23.653935,35.036106,0.170597,5.017707,42.456457,40.196202,...,0.986,146.405638,139.114222,139.165482,275.010397,151.820162,180.974035,135.565495,99.151751,98.170481
5,10-Jan-2025,42.858495,21.204845,22.089150,24.001031,35.037860,0.183182,3.974320,42.342798,40.190653,...,0.505,129.139748,140.432957,133.778931,265.095575,146.085239,172.922655,129.534301,95.266258,94.441571
6,13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,...,0.000,142.427062,132.208377,120.656198,248.032123,132.095507,153.785945,115.199220,80.115999,84.789700
7,13-Jan-2025,44.068646,13.396438,42.654157,23.997975,37.150270,0.130718,4.362098,43.682464,31.364087,...,0.000,133.775166,128.244658,116.725000,242.447284,128.060027,149.187820,111.754819,77.895613,82.220317


### Evaluate model

In [2]:
cc_model = combined_cooler.initialize()

results: list[dict] = []
Tc_in = []
Tc_out = []
for idx, ds in df.iterrows():

    mv_kgs = matlab.double([ds["mv"]/3600])  # Convert from kg/h to kg/s
    mc_kgs = matlab.double([ds["qc"]/3.6])  # Convert from m3/h to kg/s (assuming water density ~1000 kg/m3)
    Tv_C = matlab.double([ds["Tv"]])

    # Create the 'options' struct as a Python dictionary
    # options = {
    #     'model_type': 'data',
    #     'parameters': {},  # Empty struct in MATLAB
    #     'x0': float('nan'),  # Optional input as NaN
    #     'lb': matlab.double([43.324458513828800]),
    #     'ub': matlab.double([43.324458513828800]),
    # }

    tc_in, tc_out = cc_model.condenser_model(mv_kgs, Tv_C, mc_kgs, nargout=2)
    Tc_in.append(tc_in)
    Tc_out.append(tc_out)
    
results_df = pd.DataFrame({
    "Tv": df["Tv"],
    "Tc_in": Tc_in,
    "Tc_out": Tc_out,
    "qc": df["qc"],
    "mv": df["mv"],
})
results_df


Authorization required, but no authorization protocol specified



,Tv,Tc_in,Tc_out,qc,mv
0,43.324459,30.157146,39.833613,17.998552,303.464304
1,42.869820,29.272745,39.163860,17.999154,310.063340
2,43.900000,32.045439,40.013391,24.000000,333.400000
3,43.640000,31.806678,39.728931,24.000000,331.400000
4,43.082367,35.603841,40.936566,23.653935,219.734942
5,42.858495,36.168976,40.858495,24.001031,193.777819
6,44.360271,37.226856,42.360271,24.002221,214.037281
7,44.068646,37.203791,42.068646,23.997975,200.976603


In [3]:
results_df.to_csv("data_mod.csv")


### Take from pre-evaluated

In [2]:
df = pd.read_csv(data_path / "sc_out_exp.csv")
results_df = pd.read_csv(data_path / "sc_out_sim.csv")

df


,Date,qc,Tc_in,Tcond,Tc_out,mv,Tv,Tamb,Cw
0,09-Feb-2024,16.00,25.13,34.98,32.39,205.61,36.03,14.89,88.248
1,26-Feb-2024,10.00,17.07,34.73,32.30,266.93,36.09,16.70,120.600
2,16-Apr-2024,22.43,33.12,41.59,38.24,212.22,43.30,25.06,0.000
3,16-Apr-2024,24.00,33.18,41.71,37.90,209.40,43.35,27.42,140.064
4,16-Apr-2024,23.74,33.16,41.92,38.53,235.36,43.52,29.84,124.710
5,16-Apr-2024,24.00,31.66,41.49,36.80,219.49,43.44,31.02,139.218
6,18-Apr-2024,24.00,33.17,41.67,38.25,231.26,43.28,19.55,124.800
7,18-Apr-2024,24.00,33.16,41.71,38.11,216.70,43.28,20.82,130.824
8,18-Apr-2024,24.00,33.90,42.88,40.20,269.14,43.55,22.24,0.000
9,18-Apr-2024,24.00,33.68,41.99,38.83,221.89,43.43,23.09,0.000


In [5]:
from phd_visualizations.super_makers import SuperMarker

fig = regression_plot(
    df_ref= df, 
    df_mod= results_df, 
    var_ids=["Tc_out", "Tcond"], 
    units=["℃"]*2, 
    instruments=[SupportedInstruments.pt100]*2,
    show_error_metrics=True,
    var_labels=["Outlet temperature", "Condensate temperature"],
    legend_pos="top_spaced",
    inline_error_metrics_text=True,
    
    # kwargs for layout
    title_text="<b>Condenser</b> model validation",
    legend_y = 1.06, 
    title_y=0.99,
    # legend_font=dict(size=12),
    width=470,
    height=700,
    vertical_spacing=0.2,
    # margin=dict(l=10, r=10, t=200, b=10),
    
    # super_marker=SuperMarker(
    #     size_var_id="mv",
    #     size_var_range=[df.mv.min(), df.mv.max()],
    #     size_label="Vapor mass flow rate",
    # )
)
# fig.update_layout(legend_y = 1.1, legend_font=dict(size=12))

fig


In [6]:
save_figure(
    fig,
    figure_name="condenser_model_regression",
    figure_path=Path("../assets"),
    formats=["png", "svg"],
)


2025-07-13 09:08:59.839 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../assets/condenser_model_regression.png
2025-07-13 09:08:59.924 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../assets/condenser_model_regression.svg
